# 2. Notebook Overview: Session-wise Regression and Cross-session Generalization


## Notebook Order in the Pipeline

This notebook should be run **after** completing:

`1_nsd_prf_sampling_from_pyramid.ipynb`

and **before** running:

`3_Combine_Session__ROIs.ipynb`

### Purpose of This Notebook

This notebook performs the **session-wise voxel regression step** of the representational drift pipeline.

Using the pRF-sampled feature representations generated in the previous notebook, it:

* Loads pRF-pooled image features (`prfSampleLev` / `prfSampleLevOri`)
* Loads NSD single-trial beta responses
* Fits voxel-wise encoding models separately for each session
* Computes cross-session generalization matrices
* Estimates voxel-level drift metrics
* Saves regression coefficients and session prediction outputs for later analyses

### Inputs Required

## 1. NSD data

Expected under:

```text
/content/drive/MyDrive/V1_Drift/NSD
```

Required files include:

```text
nsddata/experiments/nsd/nsd_expdesign.mat
subjXX/func1pt8mm/betas_fithrf_GLMdenoise_RR/betas_sessionXX.nii.gz
subjXX/func1pt8mm/roi/prf-visualrois.nii.gz
```

The exact paths may differ depending on how the NSD folder was copied to Drive.

## 2. pRF-sampled steerable-pyramid features

Created by the previous notebook:

```text
1_nsd_prf_sampling_from_pyramid.ipynb
```

Expected under:

```text
/content/drive/MyDrive/V1_Drift/prfsample_expand
```

Typical file pattern:

```text
prfSampleStim_v{visual_region}_sub{subject}.h5
```

Example:

```text
prfSampleStim_v1_sub1.h5
```

### Main Outputs

Saved to:
`/content/drive/MyDrive/V1_Drift/repDrift_expand/`

Including:

* voxel regression coefficients
* cross-session prediction matrices
* preprocessing metadata
* split-half prediction results


This notebook corresponds most directly to the MATLAB file - `regressPrfSplit_expand.m`

from the article -

 Roth, Z.N., Merriam, E.P. Representations in human primary visual cortex drift over time. Nat Commun 14, 4422 (2023). https://doi.org/10.1038/s41467-023-40144-w

 code - https://github.com/elimerriam/repDriftNSD

and creates outputs used by later steps such as:

- ROI combination / session-combined outputs
- permutation-based drift analysis
- population-response simulation
- RDM/population-level analyses

---

## Conceptual goal

For each subject and visual region, the notebook estimates a separate encoding model for each session:

```text
pRF-sampled image features  ->  voxel beta responses
```

Then it asks:

```text
If a model is trained on session i, how well does it predict responses in session j?
```

This produces a train-session × test-session matrix. If prediction quality decreases as session distance increases, this is evidence for representational drift.

---

## Notebook organization

1. Optional Drive reset and mounting.
2. Optional image-design alignment checks.
3. Optional pRF-sampling checks.
4. Main regression pipeline.
5. Optional post-run inspection/debugging cells.

For future users, the main cell to run is the **Main regression pipeline** section. Most later cells are diagnostics and do not need to be run unless something looks wrong.


# Main outputs explained

The regression step saves one output per subject, visual region, and normalization setting.

Typical file pattern:

```text
regressPrfSplit_session_v{visual_region}_sub{subject}{normalization_suffix}.pkl
```

Examples:

```text
regressPrfSplit_session_v1_sub1.pkl
regressPrfSplit_session_v1_sub1_zeroMean.pkl
regressPrfSplit_session_v1_sub1_equalStd.pkl
```

## Important saved variables

Inside the saved dictionary, the key `nsd` contains the main regression outputs.

Common fields:

| Field | Meaning |
|---|---|
| `r2` | Within-session R² for the non-orientation/summed-orientation model |
| `r2ori` | Within-session R² for the full orientation model |
| `r2split` | Cross-session cvR² matrix for the non-orientation model |
| `r2oriSplit` | Cross-session cvR² matrix for the full orientation model |
| `pearsonR` | Cross-session Pearson r for the non-orientation model |
| `pearsonRori` | Cross-session Pearson r for the full orientation model |
| `rssSplit` | Cross-session residual sum of squares |
| `rssOriSplit` | Cross-session residual sum of squares for the orientation model |
| `voxCoef` | Session-wise fitted coefficients for the non-orientation model |
| `voxOriCoef` | Session-wise fitted coefficients for the full orientation model |
| `sessBetas` | Mean beta response per session and voxel |
| `sessStdBetas` | Standard deviation of beta responses per session and voxel |
| `splitImgTrials` | Trial mask assigning trials to sessions |
| `imgNum` | Mapping from each trial to its row in the pRF-sampled feature arrays |

## Expected matrix orientation

For cross-session matrices:

```text
[test_session, voxel, train_session]
```

or, depending on the saved helper structure:

```text
isplit, voxel, nextSplit
```

where:
- `isplit` is the session being predicted/tested.
- `nextSplit` is the session whose coefficients are used for prediction.

When plotting as train × test, check whether your plotting function transposes the matrix.


# Normalization settings

The variable `to_zscore` controls how session beta responses are normalized before regression.

| `to_zscore` | Suffix | Meaning | MATLAB equivalent |
|---:|---|---|---|
| `0` | no suffix | Raw beta responses | no normalization |
| `1` | `_zscore` | Z-score each voxel within session | `zscore(...)` |
| `2` | `_zeroMean` | Remove each voxel’s session mean | subtract mean beta |
| `3` | `_equalStd` | Equalize variance while preserving mean | subtract mean, z-score, add mean |
| `4` | `_zeroROImean` | Remove ROI-level mean | subtract ROI mean |

The article found that removing mean response amplitude strongly attenuated drift, while variance normalization did not. Therefore, `to_zscore=2` and `to_zscore=3` are especially important controls.


# Optional: Reset the Colab Drive mount

Use this only if Colab has a stale or broken `/content/drive` mount.

Important:
- This deletes only the local Colab mount directory, not your Google Drive files.
- In normal use, you can skip this cell and simply mount Drive in the next step.


In [ ]:
import shutil
import os
from google.colab import drive

mount_dir = "/content/drive"

# If the directory exists and contains files, delete it safely
if os.path.isdir(mount_dir) and os.listdir(mount_dir):
    print("Clearing /content/drive…")
    shutil.rmtree(mount_dir)

os.makedirs(mount_dir, exist_ok=True)

print("Mounting Google Drive…")
drive.mount(mount_dir, force_remount=True)


# Optional sanity check: Match `sub_design` to `allImgs`

This check verifies that the NSD trial-level image ordering can be mapped correctly onto the image IDs stored in the pRF-sampling HDF5 file.

Why this matters:
- The regression step predicts voxel responses using pRF-sampled image features.
- Each trial beta must be matched to the correct stimulus feature row.
- A mismatch here will corrupt the regression, even if the code runs without errors.


In [ ]:
import scipy.io as sio
import h5py
import numpy as np
import os
from google.colab import drive  # For Google Colab integration


# --------------------------------------------------
# Mount Google Drive
# --------------------------------------------------
print("Mounting Google Drive...")
drive.mount('/content/drive')  # Mounting Google Drive to access files
print("Google Drive mounted.")

# -----------------------
# Configuration
# -----------------------
isub = 3  # Subject number (1-based, like MATLAB)
visual_region = 1  # PRF visual region used
nsd_folder = "/content/drive/My Drive/V1_Drift/NSD"
prf_folder = "/content/drive/My Drive/V1_Drift/prfsample_expand"

# -----------------------
# Load NSD Experiment Design
# -----------------------
matfile = os.path.join(nsd_folder, "nsddata", "experiments", "nsd_expdesign.mat")
exp_design = sio.loadmat(matfile)
master_ordering = exp_design["masterordering"].squeeze() - 1  # convert to 0-based
sub_design = exp_design["subjectim"][isub - 1, master_ordering].squeeze()  # already correct after master_ordering - 1
# Do NOT do sub_design -= 1 here!

# -----------------------
# Load PRF image list (allImgs)
# -----------------------
prf_file = os.path.join(prf_folder, f"prfSampleStim_v{visual_region}_sub{isub}.h5")
with h5py.File(prf_file, "r") as f:
    all_imgs = np.array(f["allImgs"]).squeeze()  # should be 0-based and sorted

# -----------------------
# Diagnostics
# -----------------------
print(f"sub_design shape: {sub_design.shape}")
print(f"all_imgs shape: {all_imgs.shape}")

print(f"Unique sub_design images: {len(np.unique(sub_design))}")
print(f"Min/Max sub_design: {sub_design.min()} to {sub_design.max()}")
print(f"Min/Max all_imgs:   {all_imgs.min()} to {all_imgs.max()}")

# Check if all sub_design entries are in all_imgs
matches = np.isin(sub_design, all_imgs, assume_unique=True)
print(f"Matching trials: {matches.sum()} / {len(sub_design)}")

# How many sub_design images are missing from PRF set?
missing_imgs = sub_design[~matches]
print(f"Number of missing trials: {len(missing_imgs)}")
print(f"Unique missing image IDs (up to 10 shown): {np.unique(missing_imgs)[:10]}")


In [ ]:
import h5py
import pickle
import numpy as np
import os
from google.colab import drive  # For Google Colab integration


# --------------------------------------------------
# Mount Google Drive
# --------------------------------------------------
print("Mounting Google Drive...")
drive.mount('/content/drive')  # Mounting Google Drive to access files
print("Google Drive mounted.")

# -----------------------------------------------
# Update these paths according to your Colab Drive mount
# -----------------------------------------------
prf_file = "/content/drive/MyDrive/V1_Drift/prfsample_expand/prfSampleStim_v1_sub1.h5"
preprocessed_pkl = "/content/drive/MyDrive/V1_Drift/repDrift_expand/temp_preprocessed_allROIs_v1_sub1_equalStd.pkl"
roinum = 1  # ROI number you want to check (e.g., 1)

# -----------------------------------------------
# Load beta voxel indices from .pkl file
# -----------------------------------------------
print(f"Loading {preprocessed_pkl}...")
with open(preprocessed_pkl, "rb") as f:
    preproc_data = pickle.load(f)

roi_ind = preproc_data["roi_ind"]
if roinum not in roi_ind:
    raise ValueError(f"ROI {roinum} not found in roi_ind.")

# Original voxel indices used to extract betas (from NSD space)
vox_idx = roi_ind[roinum]
sorted_idx = np.argsort(vox_idx)
sorted_vox_ids = np.array(vox_idx)[sorted_idx]

print(f"Extracted {len(sorted_vox_ids)} sorted voxel indices from beta preprocessing.")

# -----------------------------------------------
# Load voxel_ids from PRF HDF5 file
# -----------------------------------------------
print(f"Loading {prf_file}...")
with h5py.File(prf_file, "r") as f:
    voxel_id_key = f"voxel_ids/roi_{roinum}"
    if voxel_id_key in f:
        prf_vox_ids = np.array(f[voxel_id_key])
        print(f"Found {len(prf_vox_ids)} voxel IDs in PRF HDF5.")
    else:
        prf_vox_ids = None
        print(f" voxel_ids/roi_{roinum} not found in {prf_file}")

# -----------------------------------------------
# Compare voxel index orderings
# -----------------------------------------------
if prf_vox_ids is None:
    print(" Cannot verify alignment — voxel_ids not found in PRF HDF5.")
else:
    if np.array_equal(sorted_vox_ids, prf_vox_ids):
        print(" Voxel ID match confirmed: beta voxels and PRF sampling aligned correctly.")
    else:
        print(" Voxel ID mismatch detected!")
        # Print differences
        diffs = np.where(sorted_vox_ids != prf_vox_ids)[0]
        print(f"First 10 mismatches (index, beta_vox, prf_vox):")
        for i in diffs[:10]:
            print(f"  {i}: {sorted_vox_ids[i]} ≠ {prf_vox_ids[i]}")


# Optional sanity check: Inspect a pRF-sampling HDF5 file

Use this section before running regression when you suspect problems in the pRF-sampling stage.

It checks whether feature arrays contain NaNs, Infs, zero-variance voxels, all-zero image patterns, or suspicious shapes.


In [ ]:
import h5py
from google.colab import drive
import os
import numpy as np

# --------------------------------------------------
# Mount Google Drive
# --------------------------------------------------
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted.")

# --------------------------------------------------
# File path and validation
# --------------------------------------------------
fpath = "/content/drive/My Drive/V1_Drift/prfsample_expand/prfSampleStim_v2_sub8.h5"

if not os.path.exists(fpath):
    raise FileNotFoundError(f"File not found: {fpath}")

# --------------------------------------------------
# Settings for the new checks
# --------------------------------------------------
EPS_STD = 1e-12          # threshold for "zero std"
SAMPLE_N_IMGS = 2000     # set None to use ALL images (can be heavy)
CHECK_ZERO_IMG = True    # check "all-zero images" pattern

def _pick_image_indices(n_images, sample_n):
    """Pick image indices to estimate std/zero-image behavior without reading everything."""
    if sample_n is None or sample_n >= n_images:
        return np.arange(n_images, dtype=np.int64)

    # Stratified: take from beginning, middle, end
    n = sample_n
    thirds = n // 3
    idx1 = np.linspace(0, max(0, n_images//3 - 1), thirds, dtype=np.int64)
    idx2 = np.linspace(n_images//3, max(n_images//3, 2*n_images//3 - 1), thirds, dtype=np.int64)
    idx3 = np.linspace(2*n_images//3, n_images - 1, n - 2*thirds, dtype=np.int64)
    idx = np.unique(np.concatenate([idx1, idx2, idx3]))
    return idx

print(f"Opening file: {fpath}")
try:
    with h5py.File(fpath, "r") as f:
        print("File opened successfully.")

        print("\n--- Checking keys ---")
        top_keys = list(f.keys())
        print("Top-level keys:", top_keys)

        # Check for required datasets
        required_keys = ["allImgs", "rois", "prfSampleLev", "prfSampleLevOri"]
        missing_keys = [k for k in required_keys if k not in f]
        if missing_keys:
            print(f"Missing expected keys: {missing_keys}")
        else:
            print("All required keys found.")

        # Validate allImgs
        n_images = None
        if "allImgs" in f:
            try:
                shape = f["allImgs"].shape
                print(f"Images dataset shape (allImgs): {shape}")
                n_images = int(shape[0])
                if n_images == 0:
                    print("Warning: 'allImgs' dataset is empty.")
            except Exception as e:
                print(f"Error reading 'allImgs': {e}")

        # Validate rois and ROI datasets + NEW checks
        if "rois" in f:
            try:
                roi_list = list(f["rois"][:])
                print(f"\nROIs found: {len(roi_list)} -> {roi_list[:20]}{'...' if len(roi_list)>20 else ''}")

                # Decide how many images to sample for checks
                if n_images is None:
                    # fallback: infer n_images from one dataset
                    # (will be overwritten below when we open first dataset)
                    pass

                for roi in roi_list:
                    roi_id = int(roi)
                    print("\n" + "="*70)
                    print(f"ROI {roi_id}")

                    # -----------------------------
                    # prfSampleLev check (3D)
                    # -----------------------------
                    lev_name = f"prfSampleLev/roi_{roi_id}"
                    if lev_name in f:
                        dset = f[lev_name]
                        print(f"{lev_name}: shape={dset.shape}, dtype={dset.dtype}")

                        if n_images is None:
                            n_images = int(dset.shape[0])

                        img_idx = _pick_image_indices(n_images, SAMPLE_N_IMGS)
                        X = dset[img_idx, ...]  # shape: [n_samp, n_vox, n_lev]

                        # NaN check (sample-based)
                        nan_count = int(np.isnan(X).sum())
                        if nan_count > 0:
                            print(f"Warning: {lev_name} sample contains NaNs (count={nan_count}).")

                        # Zero-variance check across images
                        std_vl = np.std(X, axis=0)  # [n_vox, n_lev]
                        zero_vl = std_vl < EPS_STD

                        zero_vl_count = int(zero_vl.sum())
                        total_vl = zero_vl.size
                        zero_vox_all_levels = np.all(zero_vl, axis=1)
                        zero_vox_count = int(zero_vox_all_levels.sum())

                        print(f"[STD check] sample_n_images={len(img_idx)}, EPS_STD={EPS_STD}")
                        print(f"  zero-std (voxel,level): {zero_vl_count}/{total_vl} ({100*zero_vl_count/total_vl:.4f}%)")
                        print(f"  voxels with zero std across ALL levels: {zero_vox_count}/{zero_vox_all_levels.size} ({100*zero_vox_count/zero_vox_all_levels.size:.4f}%)")

                        if zero_vox_count > 0:
                            bad_vox = np.where(zero_vox_all_levels)[0][:20]
                            print("  example constant-vox indices (first 20):", bad_vox)

                        # All-zero image check (sample-based)
                        if CHECK_ZERO_IMG:
                            mean_abs = np.mean(np.abs(X), axis=(1, 2))  # [n_samp]
                            zero_imgs = np.where(mean_abs == 0)[0]
                            if len(zero_imgs) > 0:
                                print(f"[Zero-image check] WARNING: {len(zero_imgs)}/{len(img_idx)} sampled images are all-zero in {lev_name}.")
                                print("  first few sampled positions that are all-zero:", zero_imgs[:20])
                                print("  corresponding original image indices:", img_idx[zero_imgs[:20]])
                            else:
                                print("[Zero-image check] OK: no all-zero images detected in sampled subset.")

                    else:
                        print(f"Missing dataset: {lev_name}")

                    # -----------------------------
                    # prfSampleLevOri check (4D)
                    # -----------------------------
                    ori_name = f"prfSampleLevOri/roi_{roi_id}"
                    if ori_name in f:
                        dset = f[ori_name]
                        print(f"{ori_name}: shape={dset.shape}, dtype={dset.dtype}")

                        if n_images is None:
                            n_images = int(dset.shape[0])

                        img_idx = _pick_image_indices(n_images, SAMPLE_N_IMGS)
                        Xo = dset[img_idx, ...]  # [n_samp, n_vox, n_lev, n_ori]

                        nan_count = int(np.isnan(Xo).sum())
                        if nan_count > 0:
                            print(f"Warning: {ori_name} sample contains NaNs (count={nan_count}).")

                        std_vlo = np.std(Xo, axis=0)  # [n_vox, n_lev, n_ori]
                        zero_vlo = std_vlo < EPS_STD

                        zero_vlo_count = int(zero_vlo.sum())
                        total_vlo = zero_vlo.size
                        zero_vox_all = np.all(zero_vlo, axis=(1, 2))
                        zero_vox_count = int(zero_vox_all.sum())

                        print(f"[STD check ORI] sample_n_images={len(img_idx)}, EPS_STD={EPS_STD}")
                        print(f"  zero-std (voxel,level,ori): {zero_vlo_count}/{total_vlo} ({100*zero_vlo_count/total_vlo:.4f}%)")
                        print(f"  voxels with zero std across ALL (lev,ori): {zero_vox_count}/{zero_vox_all.size} ({100*zero_vox_count/zero_vox_all.size:.4f}%)")

                        if zero_vox_count > 0:
                            bad_vox = np.where(zero_vox_all)[0][:20]
                            print("  example constant-vox indices (first 20):", bad_vox)

                        if CHECK_ZERO_IMG:
                            mean_abs = np.mean(np.abs(Xo), axis=(1, 2, 3))
                            zero_imgs = np.where(mean_abs == 0)[0]
                            if len(zero_imgs) > 0:
                                print(f"[Zero-image check ORI] WARNING: {len(zero_imgs)}/{len(img_idx)} sampled images are all-zero in {ori_name}.")
                                print("  first few sampled positions that are all-zero:", zero_imgs[:20])
                                print("  corresponding original image indices:", img_idx[zero_imgs[:20]])
                            else:
                                print("[Zero-image check ORI] OK: no all-zero images detected in sampled subset.")

                    else:
                        print(f"Missing dataset: {ori_name}")

            except Exception as e:
                print(f"Error validating ROI datasets: {e}")
        else:
            print("'rois' key not found.")

except (OSError, IOError) as e:
    print(f"Failed to open or read file: {e}")


# Optional debugging: Inspect one problematic voxel response

Use this only after the previous HDF5 check identifies a suspicious voxel or ROI.

This cell prints feature vectors for one ROI-local voxel across images and feature channels.


In [ ]:
import h5py
import numpy as np

fpath = "/content/drive/My Drive/V1_Drift/prfsample_expand/prfSampleStim_v2_sub8.h5"
roi_id = 3
vox = 356

def summarize_vector(name, v):
    v = np.asarray(v)
    finite = np.isfinite(v)
    n_finite = int(finite.sum())
    n_total = v.size
    print(f"\n{name}")
    print("-" * len(name))
    print(f"shape: {v.shape}, dtype: {v.dtype}")
    print(f"finite: {n_finite}/{n_total}, NaNs: {int(np.isnan(v).sum())}, Infs: {int(np.isinf(v).sum())}")
    if n_finite > 0:
        vv = v[finite]
        print(f"min/mean/max: {vv.min():.6g} / {vv.mean():.6g} / {vv.max():.6g}")
        print(f"std: {vv.std():.6g}")
        print(f"unique_count (approx): {len(np.unique(np.round(vv, 12)))}")
        # show first values
        print("first 10 values:", vv[:10])
        print("middle 10 values:", vv[len(vv)//2:len(vv)//2 + 10])
        print("last 10 values:", vv[-10:])
    else:
        print("No finite values to summarize.")

with h5py.File(fpath, "r") as f:
    lev_key = f"prfSampleLev/roi_{roi_id}"
    ori_key = f"prfSampleLevOri/roi_{roi_id}"

    if lev_key not in f:
        raise KeyError(f"Missing {lev_key} in file.")
    if ori_key not in f:
        raise KeyError(f"Missing {ori_key} in file.")

    # ----------------------------
    # Inspect prfSampleLev (10000, n_vox, 6)
    # ----------------------------
    X_lev = f[lev_key][:, vox, :]   # [10000, 6]
    print("Loaded:", lev_key, "->", X_lev.shape)

    # Per-level stats
    std_per_level = np.std(X_lev, axis=0)
    min_per_level = np.min(X_lev, axis=0)
    max_per_level = np.max(X_lev, axis=0)
    mean_per_level = np.mean(X_lev, axis=0)

    print("\n[prfSampleLev] per-level stats for voxel", vox)
    print("std_per_level :", std_per_level)
    print("min_per_level :", min_per_level)
    print("mean_per_level:", mean_per_level)
    print("max_per_level :", max_per_level)

    # Is it exactly constant per level?
    is_const_level = (max_per_level - min_per_level) == 0
    print("is_const_level (max-min==0):", is_const_level)

    # Overall flatten stats
    summarize_vector("[prfSampleLev] flattened (10000*6)", X_lev.reshape(-1))

    # Show a few full rows (image-wise) so you see actual values
    print("\nExample rows (image index -> 6 levels):")
    for idx in [0, 1, 2, 10, 100, 1000, 5000, 9999]:
        row = X_lev[idx]
        print(f"img {idx:4d}: {row}")

    # ----------------------------
    # Inspect prfSampleLevOri (10000, n_vox, 4, 4)
    # ----------------------------
    X_ori = f[ori_key][:, vox, :, :]   # [10000, 4, 4]
    print("\nLoaded:", ori_key, "->", X_ori.shape)

    # Per (level, ori) stats across images
    std_lo = np.std(X_ori, axis=0)     # [4,4]
    min_lo = np.min(X_ori, axis=0)
    max_lo = np.max(X_ori, axis=0)
    mean_lo = np.mean(X_ori, axis=0)

    print("\n[prfSampleLevOri] per (level,ori) std (4x4):\n", std_lo)
    print("[prfSampleLevOri] per (level,ori) min (4x4):\n", min_lo)
    print("[prfSampleLevOri] per (level,ori) mean (4x4):\n", mean_lo)
    print("[prfSampleLevOri] per (level,ori) max (4x4):\n", max_lo)

    is_const_lo = (max_lo - min_lo) == 0
    print("\nis_const_lo (max-min==0) matrix:\n", is_const_lo)

    # Flatten summary
    summarize_vector("[prfSampleLevOri] flattened (10000*4*4)", X_ori.reshape(-1))

    # Show a couple of image slices (level x ori matrix)
    print("\nExample (level x ori) matrices for a few images:")
    for idx in [0, 1, 2, 100, 1000, 5000, 9999]:
        mat = X_ori[idx]
        print(f"\nimg {idx}:\n{mat}")


In [ ]:
# next inspection: pRF params + voxel_ids for that voxel (handles your file structure)

import h5py, numpy as np

fpath = "/content/drive/My Drive/V1_Drift/prfsample_expand/prfSampleStim_v2_sub8.h5"
roi_id = 3
roi_vox = 356  # ROI-local index

with h5py.File(fpath, "r") as f:
    global_vox = int(f[f"voxel_ids/roi_{roi_id}"][roi_vox])
    print(f"ROI {roi_id} voxel {roi_vox} -> global voxel id: {global_vox}")

    for param in ["x","y","sz","r2","ecc","ang"]:
        dpath = f"prfParam/{param}/roi_{roi_id}"
        if dpath not in f:
            print(f"{param}: MISSING ({dpath})")
            continue
        arr = f[dpath][:]
        v = arr[roi_vox]
        print(f"{param}: shape={arr.shape}, voxel[{roi_vox}]={v}")



# Optional check: Inspect saved ROI Gaussian files

Use this only if you saved intermediate pRF Gaussian masks and want to verify their shape and voxel count.


In [ ]:
import pickle
import numpy as np
import sys
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Basic ROI-Gaussians inspector for files like:
#   /content/drive/My Drive/V1_Drift/prfsample_expand/roiGaussians_sub{isub}_v{vreg}.h5
# Reports voxel count per ROI and basic metadata, handling the case where each ROI
# is a TOP-LEVEL DATASET shaped (Nvox, H, W) or (H, W, Nvox).

import os, re
import h5py
import numpy as np

# ----------------------------
# Config
# ----------------------------
ISUB = 3
VREG = 1
H5_PATH = f"/content/drive/My Drive/V1_Drift/prfsample_expand/roiGaussians_sub{ISUB}_v{VREG}.h5"

if not os.path.exists(H5_PATH):
    raise FileNotFoundError(f"File not found: {H5_PATH}")

def infer_axes(shape):
    """
    Given a 3D shape, infer (Nvox, H, W).
    Prefers the two equal dims as (H,W); the remaining is Nvox.
    If ambiguous, assumes first axis is Nvox.
    """
    if len(shape) != 3:
        raise ValueError(f"Expected 3D dataset, got shape={shape}")
    a, b, c = shape
    if b == c and a != b:
        return a, b, c
    if a == b and c != a:
        return c, a, b
    if a == c and b != a:
        return b, a, c
    # Fallback
    return a, b, c  # assume Nvox=a

with h5py.File(H5_PATH, "r") as f:
    print(f"Opened: {H5_PATH}\n")

    # Find ROI datasets named 'roi_<num>'
    roi_names = [k for k in f.keys() if re.fullmatch(r"roi_\d+", k)]
    if not roi_names:
        raise RuntimeError("No ROI datasets named 'roi_<num>' found at top-level.")

    roi_names = sorted(roi_names, key=lambda s: int(s.split("_")[1]))

    total_vox = 0
    for roi_name in roi_names:
        ds = f[roi_name]
        if not isinstance(ds, h5py.Dataset):
            continue
        nvox, H, W = infer_axes(ds.shape)
        total_vox += nvox
        print(f"{roi_name}:")
        print(f"  shape = {ds.shape}  -> inferred (Nvox, H, W) = ({nvox}, {H}, {W})")
        print(f"  dtype = {ds.dtype}")

        # Light-weight spot checks on a few voxels to avoid loading the whole array
        idxs = [0, nvox // 2, nvox - 1] if nvox >= 3 else list(range(nvox))
        # Determine axis order for slicing
        a, b, c = ds.shape
        # If ds is (Nvox, H, W)
        if (a, b, c) == (nvox, H, W):
            slices = [ds[i, :, :] for i in idxs]
        # If ds is (H, W, Nvox)
        elif (a, b, c) == (H, W, nvox):
            slices = [ds[:, :, i] for i in idxs]
        else:
            # Fallback: assume first axis is Nvox
            slices = [ds[i, ...] for i in idxs]

        for i, sl in zip(idxs, slices):
            sl_min = float(np.nanmin(sl))
            sl_max = float(np.nanmax(sl))
            print(f"  voxel[{i:>4}] slice min/max = {sl_min:.6g} / {sl_max:.6g}")
        print("")

    print(f"Total ROIs: {len(roi_names)}")
    print(f"Total voxels across ROIs: {total_vox}")


# Main regression pipeline

This is the core notebook section.

It implements the Python version of the MATLAB `regressPrfSplit_expand.m` logic:

1. Load NSD single-trial beta files session by session.
2. Select voxels belonging to the requested visual region/ROI.
3. Optionally normalize beta responses per session.
4. Build one design matrix per voxel from pRF-sampled steerable-pyramid features.
5. Fit ordinary least-squares encoding models separately for each session.
6. Evaluate each session model on every other session.
7. Save within-session fits, cross-session generalization matrices, coefficients, residual summaries, and Pearson correlations.

The main outputs are saved under `SAVE_DIR`.


In [ ]:
import os
import time
import numpy as np
import nibabel as nib
import scipy.io as sio
import h5py
import pickle
import gc
from google.colab import drive


# --------------------------------------------------
# Mount Google Drive
# --------------------------------------------------
print("Mounting Google Drive...")
drive.mount('/content/drive')  # Mounting Google Drive to access files
print("Google Drive mounted.")
# -----------------------------------------------------------------------
# Paths config – adjust if needed
# -----------------------------------------------------------------------
NSD_BASE = "/content/drive/MyDrive/V1_Drift/NSD"
PRFSAMPLE_DIR = "/content/drive/MyDrive/V1_Drift/prfsample_expand"
SAVE_DIR = "/content/drive/MyDrive/V1_Drift/repDrift_expand"

os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------
def load_nifti(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return nib.load(path).get_fdata()

def rsquared(resid, orig):
    """
    MATLAB:
        r2 = 1 - sum((Xresid).^2)/sum((Xorig - mean(Xorig)).^2);
    """
    resid = np.asarray(resid, dtype=np.float64)
    orig = np.asarray(orig, dtype=np.float64)
    num = np.sum(resid ** 2)
    denom = np.sum((orig - orig.mean()) ** 2)
    if denom <= 0:
        return np.nan
    return 1.0 - num / denom

def lstsq_coeff(X, y):
    """
    Wrapper for least-squares regression: MATLAB 'X \\ y'.
    X: (n_trials, n_predictors)
    y: (n_trials,)
    Returns: (n_predictors,)
    """
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coef.astype(np.float32)


# -----------------------------------------------------------------------
# Main function: regressPrfSplit_expand → Python
# -----------------------------------------------------------------------
def regress_prf_split_expand(isub, visualRegions=None, to_zscore=0):
    """
    Python version of regressPrfSplit_expand.m

    Args:
        isub (int): subject index 1..8
        visualRegions (int or list[int]): which visualRegion group(s) to run (1..4)
        to_zscore (int): normalization flag 0–4 (same meaning as MATLAB)
    """
    if visualRegions is None:
        visualRegions = [1]
    elif isinstance(visualRegions, int):
        visualRegions = [visualRegions]

    print(f"\n=== regress_prf_split_expand: sub {isub}, visualRegions {visualRegions}, to_zscore={to_zscore} ===")
    t0_global = time.time()

    # ---------------- NSD session counts ----------------
    nsessions_sub = [40, 40, 32, 30, 40, 32, 40, 30]  # MATLAB nsessionsSub
    nsessions = 30  # nsessions_sub[isub - 1]
    nsplits = nsessions

    # ---------------- Paths ----------------
    betas_folder = os.path.join(NSD_BASE, f"subj{isub:02d}", "betas")
    nsd_expdesign_path = os.path.join(
        NSD_BASE, "nsddata", "experiments", "nsd", "nsd_expdesign.mat"
    )
    visual_rois_path = os.path.join(NSD_BASE, f"subj{isub:02d}", "func1pt8mm", "roi", "prf-visualrois.nii.gz")

    # ---------------- ROI labels ----------------
    roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]
    visRoiData = load_nifti(visual_rois_path).ravel()  # flatten like MATLAB visRoiData(:)

    # ---------------- Load NSD design ----------------
    nsd_design = sio.loadmat(nsd_expdesign_path)
    masterordering = nsd_design["masterordering"].flatten() - 1  # 0-based
    subjectim = nsd_design["subjectim"]           # shape (8, ntrials_per_subj)
    subDesign = subjectim[isub - 1, masterordering]  # trial → image ID

    # 30,000 trials total; trials per session = 10 runs × 75 trials
    ntrials = masterordering.size
    trials_per_session = 10 * 75  # 750

    # ---------------- Build splitImgTrials (nsplits × ntrials) ----------------
    splitImgTrials = np.zeros((nsplits, ntrials), dtype=np.int8)
    itrial = 0
    for isplit in range(nsplits):
        start = itrial
        end = itrial + trials_per_session
        splitImgTrials[isplit, start:end] = 1
        itrial += trials_per_session

    # ---------------- Precompute betas across sessions & ROIs ----------------
    # rois are 0-based here; visRoiData labels are 1-based (1..7), so mask with (roi+1)
    roi_indices = {1: [0, 1], 2: [2, 3], 3: [4, 5], 4: [6]}  # same grouping as MATLAB
    all_rois_needed = sorted({r for vr in visualRegions for r in roi_indices[vr]})

    # We'll build roiBetas as dict: roinum -> (nvox, ntrials_used)
    roi_betas_list = {roinum: [] for roinum in all_rois_needed}
    roiInd = {}   # roinum -> voxel indices in whole brain
    ntrials_total = 0

    print("\n-- Loading betas and building roiBetas --")
    for isession in range(1, nsessions + 1):
        # Watch the extension; change to '.nii' if your files are uncompressed
        betas_file = os.path.join(betas_folder, f"betas_session{isession:02d}.nii.gz")
        print(f"  Session {isession:02d}: {betas_file}")
        betas_4d = load_nifti(betas_file)            # (X, Y, Z, 750)
        betas_4d = betas_4d.astype(np.float32) / 300.0
        betas_2d = betas_4d.reshape(-1, betas_4d.shape[-1])  # (nvox_total, trials_per_session)
        del betas_4d
        gc.collect()

        for roinum in all_rois_needed:
            roi_mask = (visRoiData == (roinum + 1))
            roi_data = betas_2d[roi_mask, :]  # (nvox_roi, trials_per_session)

            if roinum not in roiInd:
                roiInd[roinum] = np.flatnonzero(roi_mask)

            if to_zscore == 1:
                # z-score per voxel across time
                m = roi_data.mean(axis=1, keepdims=True)
                s = roi_data.std(axis=1, ddof=0, keepdims=True)
                s[s == 0] = 1.0
                roi_norm = (roi_data - m) / s
            elif to_zscore == 2:
                # remove mean per voxel
                m = roi_data.mean(axis=1, keepdims=True)
                roi_norm = roi_data - m
            elif to_zscore == 3:
                # subtract mean, zscore, add back mean
                m = roi_data.mean(axis=1, keepdims=True)
                centered = roi_data - m
                s = centered.std(axis=1, ddof=0, keepdims=True)
                s[s == 0] = 1.0
                roi_norm = m + centered / s
            elif to_zscore == 4:
                # remove ROI grand mean
                grand_mean = roi_data.mean()
                roi_norm = roi_data - grand_mean
            else:
                roi_norm = roi_data

            roi_betas_list[roinum].append(roi_norm.astype(np.float32))

        ntrials_total += betas_2d.shape[1]
        del betas_2d
        gc.collect()

    # Concatenate across sessions: each ROI → (nvox, ntrials_used)
    roiBetas = {}
    for roinum in all_rois_needed:
        roiBetas[roinum] = np.concatenate(roi_betas_list[roinum], axis=1)
    del roi_betas_list
    gc.collect()

    # Crop splitImgTrials and subDesign/imgNum to trials we actually have betas for
    # NOTE: for subs with <40 sessions, only the first nsessions*750 trials are used
    ntrials_used = roiBetas[all_rois_needed[0]].shape[1]
    splitImgTrials = splitImgTrials[:, :ntrials_used]
    subDesign_used = subDesign[:ntrials_used]

    # -------------------------------------------------------------------
    # For each visualRegion group separately (like outer loop in MATLAB)
    # -------------------------------------------------------------------
    for visualRegion in visualRegions:
        print(f"\n=== Visual Region group: {visualRegion} ===")
        rois = roi_indices[visualRegion]  # e.g., [0,1] for V1v+V1d

        # -----------------------------------------------------
        # Load pRF samples HDF5 generated by prf_sample_model
        # -----------------------------------------------------
        prf_file = os.path.join(
            PRFSAMPLE_DIR, f"prfSampleStim_v{visualRegion}_sub{isub}.h5"
        )
        if not os.path.exists(prf_file):
            print(f"  ERROR: pRF sample file not found: {prf_file}")
            continue

        with h5py.File(prf_file, "r") as fprf:
            allImgs = fprf["allImgs"][:].astype(np.int32)
            trial_imgids = fprf["trial_imgids"][:].astype(np.int32)  # same as subDesign order
            numLevels = int(fprf.attrs["numLevels"])
            numOrientations = int(fprf.attrs["numOrientations"])

            # Map image ID → row index in prfSample arrays
            imgid_to_row = {img: i for i, img in enumerate(allImgs)}

            # imgNum: trial → index into allImgs (like MATLAB ismember(subDesign, allImgs))
            imgNum = np.full_like(subDesign_used, fill_value=-1, dtype=np.int32)
            for i, img_id in enumerate(subDesign_used):
                if img_id in imgid_to_row:
                    imgNum[i] = imgid_to_row[img_id]
            # logical imgTrials equivalent handled later via masks

            # ----------------- containers per ROI -----------------
            nvox = {}
            r2 = {}
            r2ori = {}
            r2split = {}
            r2oriSplit = {}
            rssSplit = {}
            rssOriSplit = {}
            pearsonR = {}
            pearsonRori = {}
            voxCoef = {}
            voxOriCoef = {}
            voxPredOriCoef = {}
            voxOriPredOriCoef = {}
            voxResidOriCoef = {}
            voxOriResidOriCoef = {}
            voxPredOriR2 = {}
            voxResidOriR2 = {}
            voxOriPredOriR2 = {}
            voxOriResidOriR2 = {}
            sessBetas = {}
            sessStdBetas = {}
            voxCoefCorr = {}
            voxOriCoefCorr = {}
            voxCoefCorrWithConst = {}
            voxOriCoefCorrWithConst = {}
            roiPrf = {}

            # Load roiPrf from HDF5 (optional but matches MATLAB output)
            if "prfParam" in fprf:
                for roinum in rois:
                    roiPrf[roinum] = {
                        "ecc": fprf[f"prfParam/ecc/roi_{roinum}"][:],
                        "ang": fprf[f"prfParam/ang/roi_{roinum}"][:],
                        "sz":  fprf[f"prfParam/sz/roi_{roinum}"][:],
                        "x":   fprf[f"prfParam/x/roi_{roinum}"][:],
                        "y":   fprf[f"prfParam/y/roi_{roinum}"][:],
                    }

            # -----------------------------------------------------
            # Main loop over ROIs
            # -----------------------------------------------------
            for roinum in rois:
                print(f" ROI {roinum} ({roi_names[roinum]})")

                # pRF samples: (nImages, nVox, numLevels+2 / numLevels×numOrientations)
                prfLev = fprf[f"prfSampleLev/roi_{roinum}"][:]       # float32
                prfLevOri = fprf[f"prfSampleLevOri/roi_{roinum}"][:]  # float32

                Y = roiBetas[roinum]  # (nvox, ntrials_used)
                L = Y.shape[0]
                nvox[roinum] = L

                maxNumTrials = splitImgTrials.sum(axis=1).max()

                # Allocate per-ROI arrays (matching MATLAB shapes)
                voxOriResidual = np.full((nsplits, L, maxNumTrials), np.nan, dtype=np.float32)
                voxResidual = np.full((nsplits, L, maxNumTrials), np.nan, dtype=np.float32)

                # coefficient arrays (nsplits × L × predictors)
                voxCoefSplit = np.zeros((nsplits, L, numLevels + 1 + 2), dtype=np.float32)
                voxOriCoefSplit = np.zeros((nsplits, L, numLevels * numOrientations + 1 + 2),
                                           dtype=np.float32)
                voxPredOriCoefSplit = np.zeros_like(voxOriCoefSplit)
                voxOriPredOriCoefSplit = np.zeros_like(voxOriCoefSplit)
                voxResidOriCoefSplit = np.zeros_like(voxOriCoefSplit)
                voxOriResidOriCoefSplit = np.zeros_like(voxOriCoefSplit)

                r2WithinSplit = np.zeros((nsplits, L), dtype=np.float32)
                r2oriWithinSplit = np.zeros((nsplits, L), dtype=np.float32)
                voxOriResidOriR2Split = np.zeros((nsplits, L), dtype=np.float32)
                voxOriPredOriR2Split = np.zeros((nsplits, L), dtype=np.float32)
                voxResidOriR2Split = np.zeros((nsplits, L), dtype=np.float32)
                voxPredOriR2Split = np.zeros((nsplits, L), dtype=np.float32)

                r2splitVox = np.zeros((nsplits, L, nsplits), dtype=np.float32)
                r2OrisplitVox = np.zeros((nsplits, L, nsplits), dtype=np.float32)
                rssSplitVox = np.zeros((nsplits, L, nsplits), dtype=np.float64)
                rssOrisplitVox = np.zeros((nsplits, L, nsplits), dtype=np.float64)
                pearsonRoriVox = np.zeros((nsplits, L, nsplits), dtype=np.float32)
                pearsonRVox = np.zeros((nsplits, L, nsplits), dtype=np.float32)

                sessBetasSplit = np.zeros((nsplits, L), dtype=np.float32)
                sessStdBetasSplit = np.zeros((nsplits, L), dtype=np.float32)

                # --------------- First parfor: within-split regressions ---------------
                for isplit in range(nsplits):
                    imgTrials = (splitImgTrials[isplit, :] > 0)
                    numTrials = int(imgTrials.sum())
                    if numTrials == 0:
                        continue

                    sessBetasSplit[isplit, :] = Y[:, imgTrials].mean(axis=1)
                    sessStdBetasSplit[isplit, :] = Y[:, imgTrials].std(axis=1, ddof=0)

                    trial_img_idx = imgNum[imgTrials]
                    # guard: some trials might have imgNum == -1 (no prf sample)
                    valid_mask = (trial_img_idx >= 0)
                    if not np.all(valid_mask):
                        # Drop unmatched trials from both Y and predictors
                        imgTrials_sanitized = imgTrials.copy()
                        imgTrials_sanitized[imgTrials] = valid_mask
                        imgTrials = imgTrials_sanitized
                        trial_img_idx = imgNum[imgTrials]
                        numTrials = int(imgTrials.sum())
                        if numTrials == 0:
                            continue

                    for ivox in range(L):
                        voxBetas = Y[ivox, imgTrials].astype(np.float64)  # (numTrials,)

                        # --- pRF level samples ---
                        voxPrfSample = prfLev[trial_img_idx, ivox, :]  # (numTrials, numLevels+2)
                        # add constant predictor
                        Xlev = np.concatenate(
                            [voxPrfSample, np.ones((numTrials, 1), dtype=np.float32)], axis=1
                        )  # (numTrials, numLevels+3)
                        coef_lev = lstsq_coeff(Xlev, voxBetas)
                        voxCoefSplit[isplit, ivox, :] = coef_lev

                        # --- pRF orientation samples ---
                        voxPrfOriSample = prfLevOri[trial_img_idx, ivox, :, :]  # (numTrials, numLevels, numOrient)
                        Xori_base = voxPrfOriSample.reshape(numTrials, numLevels * numOrientations, order="F")

                        # add lowest & highest SF (taken from Xlev cols end-2:end-1)
                        low_high = Xlev[:, -3:-1]  # those are original low/high SF predictors
                        Xori_ext = np.concatenate([Xori_base, low_high], axis=1)

                        # add constant
                        Xori = np.concatenate(
                            [Xori_ext, np.ones((numTrials, 1), dtype=np.float32)], axis=1
                        )  # (numTrials, numLevels*numOrient+3)
                        coef_ori = lstsq_coeff(Xori, voxBetas)
                        voxOriCoefSplit[isplit, ivox, :] = coef_ori

                        # --- regress vignetting model prediction with orientation model ---
                        voxPred = Xlev @ coef_lev  # vignetting model prediction
                        coef_predOri = lstsq_coeff(Xori, voxPred)
                        voxPredOriCoefSplit[isplit, ivox, :] = coef_predOri

                        # --- regress full orientation model prediction with orientation model ---
                        voxOriPred = Xori @ coef_ori
                        coef_oriPredOri = lstsq_coeff(Xori, voxOriPred)
                        voxOriPredOriCoefSplit[isplit, ivox, :] = coef_oriPredOri

                        # --- residuals within split ---
                        voxOriResidual_vec = voxBetas - Xori @ coef_ori
                        voxResidual_vec = voxBetas - Xlev @ coef_lev

                        voxOriResidual[isplit, ivox, :numTrials] = voxOriResidual_vec.astype(np.float32)
                        voxResidual[isplit, ivox, :numTrials] = voxResidual_vec.astype(np.float32)

                        # regress residuals of vignetting model with full model
                        coef_residOri = lstsq_coeff(Xori, voxResidual_vec)
                        voxResidOriCoefSplit[isplit, ivox, :] = coef_residOri

                        # regress residuals of orientation model with orientation model
                        coef_oriResidOri = lstsq_coeff(Xori, voxOriResidual_vec)
                        voxOriResidOriCoefSplit[isplit, ivox, :] = coef_oriResidOri

                        # R² within split
                        r2WithinSplit[isplit, ivox] = rsquared(
                            voxResidual_vec, Y[ivox, imgTrials]
                        )
                        r2oriWithinSplit[isplit, ivox] = rsquared(
                            voxOriResidual_vec, Y[ivox, imgTrials]
                        )

                        # residuals of full orientation model after oriResidOri fit
                        oriResidOriResidual = voxBetas - (Xori @ coef_oriResidOri)
                        voxOriResidOriR2Split[isplit, ivox] = rsquared(
                            oriResidOriResidual, Y[ivox, imgTrials]
                        )

                        # residuals of full orientation model after oriPredOri fit
                        oriPredOriResidual = voxBetas - (Xori @ coef_oriPredOri)
                        voxOriPredOriR2Split[isplit, ivox] = rsquared(
                            oriPredOriResidual, Y[ivox, imgTrials]
                        )

                        # residuals of constrained model (predicted residuals of vignetting model)
                        residOriResidual = voxBetas - (Xori @ coef_residOri)
                        voxResidOriR2Split[isplit, ivox] = rsquared(
                            residOriResidual, Y[ivox, imgTrials]
                        )

                        # residuals of constrained model (prediction of constrained model)
                        predOriResidual = voxBetas - (Xori @ coef_predOri)
                        voxPredOriR2Split[isplit, ivox] = rsquared(
                            predOriResidual, Y[ivox, imgTrials]
                        )

                # --------------- Second parfor: cross-split metrics ---------------
                for isplit in range(nsplits):
                    imgTrials = (splitImgTrials[isplit, :] > 0)
                    numTrials = int(imgTrials.sum())
                    if numTrials == 0:
                        continue

                    trial_img_idx = imgNum[imgTrials]
                    valid_mask = (trial_img_idx >= 0)
                    if not np.all(valid_mask):
                        imgTrials_sanitized = imgTrials.copy()
                        imgTrials_sanitized[imgTrials] = valid_mask
                        imgTrials = imgTrials_sanitized
                        trial_img_idx = imgNum[imgTrials]
                        numTrials = int(imgTrials.sum())
                        if numTrials == 0:
                            continue

                    Xlev = prfLev[trial_img_idx, :, :]  # (numTrials, L, numLevels+2)
                    Xori_all = prfLevOri[trial_img_idx, :, :, :]  # (numTrials, L, numLevels, numOrient)

                    for ivox in range(L):
                        voxBetas = Y[ivox, imgTrials].astype(np.float64)  # (numTrials,)

                        # design matrices for this voxel & split
                        Xlev_vox = Xlev[:, ivox, :]  # (numTrials, numLevels+2)
                        Xlev_with_const = np.concatenate(
                            [Xlev_vox, np.ones((numTrials, 1), dtype=np.float32)], axis=1
                        )

                        Xori_vox = Xori_all[:, ivox, :, :].reshape(
                            numTrials, numLevels * numOrientations, order="F"
                        )
                        low_high = Xlev_with_const[:, -3:-1]
                        Xori_ext = np.concatenate([Xori_vox, low_high], axis=1)
                        Xori_with_const = np.concatenate(
                            [Xori_ext, np.ones((numTrials, 1), dtype=np.float32)], axis=1
                        )

                        for nextSplit in range(nsplits):
                            coef_lev_next = voxCoefSplit[nextSplit, ivox, :]
                            coef_ori_next = voxOriCoefSplit[nextSplit, ivox, :]

                            voxResidualSplit_vec = voxBetas - Xlev_with_const @ coef_lev_next
                            voxOriResidualSplit_vec = voxBetas - Xori_with_const @ coef_ori_next

                            r2splitVox[isplit, ivox, nextSplit] = rsquared(
                                voxResidualSplit_vec, voxBetas
                            )
                            r2OrisplitVox[isplit, ivox, nextSplit] = rsquared(
                                voxOriResidualSplit_vec, voxBetas
                            )

                            rssSplitVox[isplit, ivox, nextSplit] = np.sum(
                                voxResidualSplit_vec ** 2
                            )
                            rssOrisplitVox[isplit, ivox, nextSplit] = np.sum(
                                voxOriResidualSplit_vec ** 2
                            )

                            # Pearson correlation between voxBetas and predictions
                            y_pred_ori = Xori_with_const @ coef_ori_next
                            y_pred_lev = Xlev_with_const @ coef_lev_next
                            if np.all(np.isfinite(y_pred_ori)) and np.std(y_pred_ori) > 0:
                                pearsonRoriVox[isplit, ivox, nextSplit] = np.corrcoef(
                                    voxBetas, y_pred_ori
                                )[0, 1]
                            else:
                                pearsonRoriVox[isplit, ivox, nextSplit] = np.nan

                            if np.all(np.isfinite(y_pred_lev)) and np.std(y_pred_lev) > 0:
                                pearsonRVox[isplit, ivox, nextSplit] = np.corrcoef(
                                    voxBetas, y_pred_lev
                                )[0, 1]
                            else:
                                pearsonRVox[isplit, ivox, nextSplit] = np.nan

                # store per-ROI results
                sessBetas[roinum] = sessBetasSplit
                sessStdBetas[roinum] = sessStdBetasSplit

                voxPredOriR2[roinum] = voxPredOriR2Split
                voxResidOriR2[roinum] = voxResidOriR2Split
                voxOriPredOriR2[roinum] = voxOriPredOriR2Split
                voxOriResidOriR2[roinum] = voxOriResidOriR2Split

                r2ori[roinum] = r2oriWithinSplit
                r2[roinum] = r2WithinSplit

                voxResidOriCoef[roinum] = voxResidOriCoefSplit
                voxOriResidOriCoef[roinum] = voxOriResidOriCoefSplit
                voxOriCoef[roinum] = voxOriCoefSplit
                voxCoef[roinum] = voxCoefSplit
                voxPredOriCoef[roinum] = voxPredOriCoefSplit
                voxOriPredOriCoef[roinum] = voxOriPredOriCoefSplit

                r2split[roinum] = r2splitVox
                r2oriSplit[roinum] = r2OrisplitVox
                rssOriSplit[roinum] = rssOrisplitVox
                rssSplit[roinum] = rssSplitVox
                pearsonRori[roinum] = pearsonRoriVox
                pearsonR[roinum] = pearsonRVox

                # ---- correlate coefficients between splits (with & without constant) ----
                voxCoefCorr[roinum] = np.zeros((L, nsplits, nsplits), dtype=np.float32)
                voxOriCoefCorr[roinum] = np.zeros((L, nsplits, nsplits), dtype=np.float32)
                voxCoefCorrWithConst[roinum] = np.zeros((L, nsplits, nsplits), dtype=np.float32)
                voxOriCoefCorrWithConst[roinum] = np.zeros((L, nsplits, nsplits), dtype=np.float32)

                for ivox in range(L):
                    # without constant (drop last predictor)
                    coef_lev_no_const = voxCoefSplit[:, ivox, :-1]      # (nsplits, P)
                    coef_ori_no_const = voxOriCoefSplit[:, ivox, :-1]   # (nsplits, P')

                    # with constant
                    coef_lev_all = voxCoefSplit[:, ivox, :]
                    coef_ori_all = voxOriCoefSplit[:, ivox, :]

                    voxCoefCorr[roinum][ivox, :, :] = np.corrcoef(coef_lev_no_const) if coef_lev_no_const.shape[0] > 1 else np.eye(nsplits)
                    voxOriCoefCorr[roinum][ivox, :, :] = np.corrcoef(coef_ori_no_const) if coef_ori_no_const.shape[0] > 1 else np.eye(nsplits)
                    voxCoefCorrWithConst[roinum][ivox, :, :] = np.corrcoef(coef_lev_all) if coef_lev_all.shape[0] > 1 else np.eye(nsplits)
                    voxOriCoefCorrWithConst[roinum][ivox, :, :] = np.corrcoef(coef_ori_all) if coef_ori_all.shape[0] > 1 else np.eye(nsplits)

            # ---------------- wrap into nsd dict (like MATLAB struct) ----------------
            nsd = {
                "voxOriCoefCorrWithConst": voxOriCoefCorrWithConst,
                "voxCoefCorrWithConst": voxCoefCorrWithConst,
                "voxOriCoefCorr": voxOriCoefCorr,
                "voxCoefCorr": voxCoefCorr,
                "r2": r2,
                "r2ori": r2ori,
                "r2split": r2split,
                "r2oriSplit": r2oriSplit,
                "rssOriSplit": rssOriSplit,
                "rssSplit": rssSplit,
                "pearsonRori": pearsonRori,
                "pearsonR": pearsonR,
                "imgNum": imgNum,
                "splitImgTrials": splitImgTrials,
                "voxCoef": voxCoef,
                "voxOriCoef": voxOriCoef,
                "voxPredOriCoef": voxPredOriCoef,
                "voxOriPredOriCoef": voxOriPredOriCoef,
                "voxResidOriCoef": voxResidOriCoef,
                "voxOriResidOriCoef": voxOriResidOriCoef,
                "voxPredOriR2": voxPredOriR2,
                "voxOriPredOriR2": voxOriPredOriR2,
                "voxResidOriR2": voxResidOriR2,
                "voxOriResidOriR2": voxOriResidOriR2,
                "sessBetas": sessBetas,
                "sessStdBetas": sessStdBetas,
                "roiInd": roiInd,
            }

            # zscore suffix for filename
            if to_zscore == 1:
                zscore_str = "_zscore"
            elif to_zscore == 2:
                zscore_str = "_zeroMean"
            elif to_zscore == 3:
                zscore_str = "_equalStd"
            elif to_zscore == 4:
                zscore_str = "_zeroROImean"
            else:
                zscore_str = ""

            out_name = f"regressPrfSplit_session_v{visualRegion}_sub{isub}{zscore_str}.pkl"
            out_path = os.path.join(SAVE_DIR, out_name)
            print(f"  Saving results to {out_path}")
            with open(out_path, "wb") as f_out:
                pickle.dump(
                    {
                        "nsd": nsd,
                        "numLevels": numLevels,
                        "numOrientations": numOrientations,
                        "rois": rois,
                        "nvox": nvox,
                        "roiPrf": roiPrf,
                        "nsplits": nsplits,
                        "roi_names": roi_names,
                    },
                    f_out,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )

    t1_global = time.time()
    print(f"\nDone regress_prf_split_expand in {t1_global - t0_global:.1f} s.")


# -----------------------------------------------------------------------
# Example call (like MATLAB: regressPrfSplit_expand(isubject, 1, 0))
# -----------------------------------------------------------------------
regress_prf_split_expand(isub=2, visualRegions=[1, 2, 3, 4], to_zscore=0)
regress_prf_split_expand(isub=3, visualRegions=[1, 2, 3, 4], to_zscore=0)
regress_prf_split_expand(isub=4, visualRegions=[1, 2, 3, 4], to_zscore=0)
regress_prf_split_expand(isub=5, visualRegions=[1, 2, 3, 4], to_zscore=0)
regress_prf_split_expand(isub=6, visualRegions=[1, 2, 3, 4], to_zscore=0)
regress_prf_split_expand(isub=7, visualRegions=[1, 2, 3, 4], to_zscore=0)
regress_prf_split_expand(isub=8, visualRegions=[1, 2, 3, 4], to_zscore=0)



# Minimal checklist after the main regression finishes

Before continuing to the permutation or population-analysis notebooks, confirm:

- The output file exists in `SAVE_DIR`.
- `r2split` and `r2oriSplit` have the expected shape.
- Diagonal values are generally higher than off-diagonal values.
- Pearson-r values are finite.
- No ROI has an unexpected voxel count of zero.
- The filename suffix matches the normalization you intended to run.


import pickle, numpy as np, os

path = "/content/drive/MyDrive/V1_Drift/repDrift_expand/regressPrfSplit_session_v1_sub1.pkl"

with open(path, "rb") as f:
    D = pickle.load(f)

nsd = D["nsd"]

for roi in nsd["r2"]:
    print("ROI", roi)
    print("mean r2 non-ori:", np.nanmean(nsd["r2"][roi]))
    print("mean r2 ori:    ", np.nanmean(nsd["r2ori"][roi]))
    print("mean pearson non-ori:", np.nanmean(nsd["pearsonR"][roi]))
    print("mean pearson ori:    ", np.nanmean(nsd["pearsonRori"][roi]))
    print()

# Post-run inspection: Inspect a regression output pickle

Use this after running the main regression cell.

This section checks saved arrays, shapes, mean within-session fit, mean cross-session generalization, and the difference between orientation and non-orientation models.


In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

# --------------------------------------------------
# Mount Google Drive (safe)
# --------------------------------------------------
if not os.path.ismount("/content/drive"):
    print("Mounting Google Drive...")
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

# --------------------------------------------------
# Load the pickle file
# --------------------------------------------------
file_path = "/content/drive/MyDrive/V1_Drift/repDrift_expand/regressPrfSplit_session_v1_sub8.pkl"
assert os.path.exists(file_path), f"File not found: {file_path}"
print(f"\nLoading {file_path} ...")

with open(file_path, "rb") as f:
    data = pickle.load(f)

nsd = data["nsd"]
rois = data["rois"]
roi_names = data["roi_names"]
nsplits = data["nsplits"]
numLevels = data["numLevels"]
numOrientations = data["numOrientations"]
nvox = data["nvox"]

print("\n=== Top-level metadata ===")
print("rois:", rois)
print("roi_names:", roi_names)
print("nsplits:", nsplits)
print("numLevels:", numLevels)
print("numOrientations:", numOrientations)
print("nvox:", nvox)

# ----------------------------
# Helpers
# ----------------------------
def finite_stats(arr, quantiles=(0.05, 0.5, 0.95)):
    """Return stats over finite values only (excludes NaN/±Inf)."""
    arr = np.asarray(arr)
    finite = np.isfinite(arr)
    n = arr.size
    n_fin = int(finite.sum())
    n_nan = int(np.isnan(arr).sum())
    n_inf = int(np.isinf(arr).sum())
    pct_nan = 100.0 * n_nan / n if n else 0.0
    pct_inf = 100.0 * n_inf / n if n else 0.0

    if n_fin == 0:
        return {
            "n": n, "n_fin": 0, "n_nan": n_nan, "n_inf": n_inf,
            "pct_nan": pct_nan, "pct_inf": pct_inf,
            "mean": None, "std": None, "min": None, "max": None, "q": None
        }

    vals = arr[finite].astype(np.float64, copy=False)
    qvals = np.quantile(vals, quantiles) if len(vals) > 0 else [None] * len(quantiles)
    return {
        "n": n, "n_fin": n_fin, "n_nan": n_nan, "n_inf": n_inf,
        "pct_nan": pct_nan, "pct_inf": pct_inf,
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
        "q": [float(x) for x in qvals]
    }

def print_stats_line(name, arr, extra=None):
    s = finite_stats(arr)
    q = s["q"]
    if q and s["n_fin"]:
        qtxt = f" | q05={q[0]:.4g}, q50={q[1]:.4g}, q95={q[2]:.4g}"
    else:
        qtxt = " | q05/50/95=NA"

    mean_str = f"{s['mean']:.4g}" if s["mean"] is not None else "NA"
    std_str = f"{s['std']:.4g}" if s["std"] is not None else "NA"
    min_str = f"{s['min']:.4g}" if s["min"] is not None else "NA"
    max_str = f"{s['max']:.4g}" if s["max"] is not None else "NA"

    print(
        f"{name:<35} shape={arr.shape}  "
        f"fin={s['n_fin']}/{s['n']}  NaN={s['pct_nan']:.2f}%  Inf={s['pct_inf']:.2f}%  "
        f"mean={mean_str}  std={std_str}  min={min_str}  max={max_str}"
        f"{qtxt}"
        + (f"  {extra}" if extra else "")
    )

def offdiag_mask(shape):
    """Boolean mask for off-diagonal entries in (S,S) matrices."""
    assert len(shape) >= 2 and shape[-1] == shape[-2]
    S = shape[-1]
    m = np.ones((S, S), dtype=bool)
    np.fill_diagonal(m, False)
    return m

# --------------------------------------------------
# Quick stats for all nsd keys
# --------------------------------------------------
print("\n=== nsd keys (types) ===")
for key, val in nsd.items():
    if isinstance(val, dict):
        print(f"{key:<25} dict with ROIs: {list(val.keys())}")
    else:
        print_stats_line(key, np.asarray(val))

# --------------------------------------------------
# Detailed per-ROI stats for key metrics
# --------------------------------------------------
keys_main = [
    "r2", "r2ori",
    "r2split", "r2oriSplit",
    "rssSplit", "rssOriSplit",
    "pearsonR", "pearsonRori",
    "voxCoefCorr", "voxOriCoefCorr",
    "voxCoefCorrWithConst", "voxOriCoefCorrWithConst",
]

for roi in rois:
    print(f"\n================ ROI {roi} ({roi_names[roi]}) ================")

    for k in keys_main:
        if k not in nsd:
            continue
        val = nsd[k]
        if not isinstance(val, dict) or roi not in val:
            continue

        arr = np.asarray(val[roi])

        # 3D: (S, L, S) e.g. r2split, pearsonR
        if arr.ndim == 3 and arr.shape[0] == arr.shape[2]:
            S, L, _ = arr.shape
            print_stats_line(k, arr)

            # diagonal (same split train==test)
            diag_vals = np.empty((L, S), dtype=float)
            for s in range(S):
                diag_vals[:, s] = arr[s, :, s]
            print_stats_line(k + " (diag over splits)", diag_vals)

            # off-diagonal (different splits)
            off_list = []
            for s in range(S):
                for t in range(S):
                    if s == t:
                        continue
                    off_list.append(arr[s, :, t])
            off_vals = np.stack(off_list, axis=1) if off_list else np.empty((L, 0))
            print_stats_line(k + " (offdiag over splits)", off_vals)

        # 3D: (V, S, S) e.g. voxCoefCorr
        elif arr.ndim == 3 and arr.shape[1] == arr.shape[2]:
            V, S, _ = arr.shape
            print_stats_line(k, arr)

            m_off = offdiag_mask((S, S))
            off_vals = arr[:, m_off].reshape(V, -1)
            print_stats_line(k + " (offdiag only)", off_vals)

            diag_vals = np.diagonal(arr, axis1=1, axis2=2)  # (V, S)
            print_stats_line(k + " (diag only)", diag_vals)

        else:
            # 1D/2D metrics (r2, r2ori, etc.)
            print_stats_line(k, arr)

# --------------------------------------------------
# Example: coefficient-correlation heatmap for one voxel
# --------------------------------------------------
roi_example = rois[0]          # e.g. first ROI (0 -> V1v)
ivox = 100                      # voxel index to display
key_corr = "voxCoefCorr"

if key_corr in nsd and roi_example in nsd[key_corr]:
    corr = nsd[key_corr][roi_example][ivox]  # (S, S)
    print(f"\nVoxel {ivox} in ROI {roi_example} ({roi_names[roi_example]})")
    print("  corr shape:", corr.shape)

    s_all = finite_stats(corr)
    print(f"  mean r (finite-only) = {s_all['mean'] if s_all['mean'] is not None else 'NA'}")

    if corr.shape[0] >= 6:
        row = corr[5, :]
        row_fin = row[np.isfinite(row)]
        print("  Row 6 finite mean:", np.mean(row_fin) if row_fin.size else 'NA')
        print("  Row 6 values (first 10):", np.round(row[:10], 3))

    masked = np.ma.masked_invalid(corr)
    plt.figure(figsize=(5, 4))
    im = plt.imshow(masked, interpolation="nearest")
    plt.title(f"ROI {roi_example} — voxel {ivox} coef corr (masked invalid)")
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()
else:
    print(f"\nNo '{key_corr}' data available for ROI {roi_example}.")


r2 and r2ori are within-session model fits.

r2split and r2oriSplit are cross-session generalization matrices.

* **r2split**: The full train-session × test-session matrix of cross-validated R² values for each voxel. It includes both within-session fits (train = test) and cross-session generalization (train ≠ test).

* **r2split (diag over splits)**: The diagonal of that matrix (train = test). This reflects the model’s within-session goodness-of-fit — essentially standard encoding performance per session.

* **r2split (offdiag over splits)**: All off-diagonal entries (train ≠ test). This measures cross-session generalization and is used to quantify representational stability or drift over time.


r2oriSplit: The full train-session × test-session matrix of cross-validated R² values computed using the orientation-level feature space (i.e., keeping orientation channels separate rather than collapsing them).

r2oriSplit (diag over splits): Within-session fit (train = test) for the orientation-resolved model. This reflects standard encoding performance per session using orientation features.

r2oriSplit (offdiag over splits): Cross-session generalization (train ≠ test) for the orientation-resolved model. This is what you use to assess representational stability / drift when orientation information is preserved.

RSS = Residual Sum of Squares

So unlike R² (which is variance explained), Pearson r measures:
Linear similarity between predicted and actual responses.